# 🎯 CodeOptimizer — Success Probability Predictor
## Cold-Start ML Model | Training, Evaluation & Scheduled Retraining Pipeline

This notebook covers the **full ML lifecycle** for predicting whether a user will successfully solve a LeetCode problem before they attempt it:

1. **Data Loading & Exploration** — from MongoDB-exported JSON files  
2. **Feature Engineering** — user × problem interaction features  
3. **Model Training** — Logistic Regression + Gradient Boosting ensemble  
4. **Evaluation & Interpretability** — CV scores, coefficients, confusion matrix  
5. **Prediction Function** — ready-to-call from your FastAPI backend  
6. **Scheduled Retraining** — interval-based pipeline using `APScheduler`

> **Plug the data paths** at the top of Section 1 and run all cells. The trained models are saved as `.pkl` files that the FastAPI predictor service loads at startup.

## 0 · Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import os
import pickle
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Data & ML ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, train_test_split
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Scheduler (pip install apscheduler) ───────────────────────────────────────
try:
    from apscheduler.schedulers.background import BackgroundScheduler
    SCHEDULER_AVAILABLE = True
except ImportError:
    SCHEDULER_AVAILABLE = False
    print("APScheduler not installed. Run: pip install apscheduler")
    print("Retraining scheduler cell will be skipped.")

print(f"NumPy  {np.__version__}")
print(f"Pandas {pd.__version__}")
print("All imports successful ✅")

## 1 · Paths & Constants

> **Edit `DATA_DIR` and `MODEL_DIR`** to match your project layout before running.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path("./data")          # folder containing the three JSON exports
MODEL_DIR = Path("./models")        # where trained .pkl files will be saved
MODEL_DIR.mkdir(parents=True, exist_ok=True)

USERS_FILE    = DATA_DIR / "users.json"
INTERACTIONS_FILE = DATA_DIR / "interactions.json"
PROBLEMS_FILE = DATA_DIR / "display-problems.json"

# ── Encoding maps ─────────────────────────────────────────────────────────────
DIFFICULTY_MAP  = {"Easy": 1, "Medium": 2, "Hard": 3}
EXPERIENCE_MAP  = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}

# ── Model hyperparameters ─────────────────────────────────────────────────────
LR_PARAMS  = dict(max_iter=500, random_state=42, class_weight="balanced")
GBM_PARAMS = dict(n_estimators=200, learning_rate=0.05,
                  max_depth=4, subsample=0.8, random_state=42)

# ── Feature columns (order matters for pickling) ──────────────────────────────
FEATURE_COLS = [
    "userRating",      # Elo-style rating
    "accuracy",        # overall submission accuracy  [0, 1]
    "solvedProblems",  # total problems solved
    "experience",      # encoded: Beginner=1 / Intermediate=2 / Advanced=3
    "skillMatch",      # avg skill level across problem's topic tags  [0, 1]
    "difficulty",      # encoded: Easy=1 / Medium=2 / Hard=3
    "acceptanceRate",  # LeetCode global acceptance rate  [0, 1]
]

print("Paths configured:")
print(f"  Data  → {DATA_DIR.resolve()}")
print(f"  Models → {MODEL_DIR.resolve()}")

## 2 · Data Loading & Exploration

In [ ]:
def load_json(path: Path) -> list | dict:
    """Load a JSON file; raise a clear error if missing."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Export your MongoDB collections to JSON and update DATA_DIR."
        )
    with open(path) as f:
        return json.load(f)

users        = load_json(USERS_FILE)
interactions = load_json(INTERACTIONS_FILE)
problems     = load_json(PROBLEMS_FILE)

print(f"Users loaded        : {len(users)}")
print(f"Interactions loaded : {len(interactions)}")
print(f"Problems loaded     : {len(problems)}")

In [ ]:
# Quick schema preview
print("── User sample fields ──────────────────────────────")
u = users[0]
for k, v in u.items():
    if k not in ("recentActivity", "skillDistribution", "password", "avatar"):
        print(f"  {k}: {v}")

print("\n── Interaction sample ──────────────────────────────")
ix = interactions[0]
for k, v in ix.items():
    print(f"  {k}: {v}")

In [ ]:
# Label distribution across all interactions
labels = [1 if ix.get("submissionStatus") == 1 else 0 for ix in interactions]
total  = len(labels)
solved = sum(labels)
print(f"Total interactions : {total}")
print(f"Solved (label=1)   : {solved}  ({solved/total:.1%})")
print(f"Failed (label=0)   : {total-solved}  ({(total-solved)/total:.1%})")

## 3 · Feature Engineering

Each training row joins one **interaction** with the matching **user snapshot** and **problem metadata**.

| Feature | Logic |
|---|---|
| `userRating` | Elo-style numeric rating stored on the user document |
| `accuracy` | `correctProblems / totalInteractions` at time of interaction |
| `solvedProblems` | Count of unique solved problem IDs |
| `experience` | Encoded ordinal: Beginner=1, Intermediate=2, Advanced=3 |
| `skillMatch` | Mean skill level over the problem's topic tags from the user's `skillDistribution` vector |
| `difficulty` | Encoded ordinal: Easy=1, Medium=2, Hard=3 |
| `acceptanceRate` | LeetCode global acceptance ÷ 100 (normalised to [0,1]) |
| `label` | 1 = `submissionStatus == 1` (solved), 0 = failed/attempted |

In [ ]:
def compute_skill_match(skill_distribution: list, tags: list) -> float:
    """
    Compute average user skill level across the problem's topic tags.
    Falls back to 0.1 if no matching tags are found.
    """
    if not skill_distribution or not tags:
        return 0.1
    skill_map = {s["name"]: s["level"] for s in skill_distribution}
    levels = [skill_map.get(tag, 0.0) for tag in tags]
    return float(np.mean(levels)) if levels else 0.1


def build_feature_dataframe(
    users: list,
    interactions: list,
    problems: list
) -> pd.DataFrame:
    """
    Join interactions with user and problem data to build the training table.
    Rows with missing user or problem references are silently dropped.
    """
    user_lookup    = {u["userId"]: u for u in users}
    problem_lookup = {p["id"]: p for p in problems}

    rows = []
    skipped = 0

    for ix in interactions:
        uid  = ix.get("userId")
        pid  = str(ix.get("problemId", ""))
        user = user_lookup.get(uid)
        prob = problem_lookup.get(pid)

        if not user or not prob:
            skipped += 1
            continue

        skill_match = compute_skill_match(
            user.get("skillDistribution", []),
            prob.get("tags", [])
        )

        rows.append({
            # ── User features ─────────────────────────────────────────
            "userRating":       user.get("userRating", 1200),
            "accuracy":         user.get("accuracy", 0.5),
            "solvedProblems":   user.get("solvedProblems", 0),
            "experience":       EXPERIENCE_MAP.get(user.get("experience", "Beginner"), 1),
            # ── Interaction-derived ────────────────────────────────────
            "skillMatch":       skill_match,
            # ── Problem features ──────────────────────────────────────
            "difficulty":       DIFFICULTY_MAP.get(prob.get("difficulty", "Medium"), 2),
            "acceptanceRate":   prob.get("acceptanceRate", 50.0) / 100.0,
            # ── Label ─────────────────────────────────────────────────
            "label":            1 if ix.get("submissionStatus") == 1 else 0,
            # ── Metadata (not used in training) ───────────────────────
            "_userId":    uid,
            "_problemId": pid,
            "_title":     prob.get("title", ""),
            "_difficulty": prob.get("difficulty", ""),
        })

    df = pd.DataFrame(rows)
    print(f"Feature rows built : {len(df)}  (skipped {skipped} unmatched rows)")
    return df


df = build_feature_dataframe(users, interactions, problems)
df.head()

In [ ]:
# Descriptive statistics for the feature columns
df[FEATURE_COLS + ["label"]].describe().round(4)

In [ ]:
# Per-difficulty class balance
pivot = df.groupby(["_difficulty", "label"]).size().unstack(fill_value=0)
pivot.columns = ["Failed", "Solved"]
pivot["SuccessRate"] = (pivot["Solved"] / pivot.sum(axis=1)).round(3)
print(pivot)

## 4 · Model Training

We train two complementary models and ensemble their probability outputs:

- **Logistic Regression** — fast, interpretable, good on small data  
- **Gradient Boosting** — higher accuracy, captures non-linear interactions  

Both are wrapped in a `Pipeline` with `StandardScaler` to handle the mixed-scale features.

In [ ]:
X = df[FEATURE_COLS].values
y = df["label"].values

# Reproducible train/test split for final evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {len(X_train)} rows  |  Test : {len(X_test)} rows")
print(f"Train positive rate : {y_train.mean():.1%}")
print(f"Test  positive rate : {y_test.mean():.1%}")

In [ ]:
# ── Build pipelines ───────────────────────────────────────────────────────────
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr",     LogisticRegression(**LR_PARAMS))
])

gbm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("gbm",    GradientBoostingClassifier(**GBM_PARAMS))
])

# ── Cross-validation ──────────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=min(5, len(X_train)), shuffle=True, random_state=42)

for name, pipe in [("Logistic Regression", lr_pipeline), ("Gradient Boosting", gbm_pipeline)]:
    accs = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    aucs = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}")
    print(f"  Accuracy : {accs.mean():.3f} ± {accs.std():.3f}")
    print(f"  ROC-AUC  : {aucs.mean():.3f} ± {aucs.std():.3f}")
    print()

In [ ]:
# ── Final fit on full training set ────────────────────────────────────────────
lr_pipeline.fit(X_train, y_train)
gbm_pipeline.fit(X_train, y_train)
print("Models trained ✅")

## 5 · Evaluation & Interpretability

In [ ]:
# ── Hold-out test set evaluation ──────────────────────────────────────────────
lr_probs   = lr_pipeline.predict_proba(X_test)[:, 1]
gbm_probs  = gbm_pipeline.predict_proba(X_test)[:, 1]
ens_probs  = (lr_probs + gbm_probs) / 2
ens_preds  = (ens_probs >= 0.5).astype(int)

print("── Ensemble (LR + GBM average) ────────────────────")
print(f"Accuracy : {accuracy_score(y_test, ens_preds):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, ens_probs):.3f}")
print()
print(classification_report(y_test, ens_preds, target_names=["Failed", "Solved"]))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Confusion Matrices", fontsize=13, fontweight="bold")

for ax, (name, probs) in zip(axes, [
    ("Logistic Regression", lr_probs),
    ("Gradient Boosting",   gbm_probs),
    ("Ensemble",            ens_probs),
]):
    preds = (probs >= 0.5).astype(int)
    cm    = confusion_matrix(y_test, preds)
    disp  = ConfusionMatrixDisplay(cm, display_labels=["Failed", "Solved"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}\nAcc={accuracy_score(y_test, preds):.2f}", fontsize=11)

plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → models/confusion_matrices.png")

In [ ]:
# ── LR Feature coefficients (interpretability) ────────────────────────────────
lr_model   = lr_pipeline.named_steps["lr"]
coef_df    = pd.DataFrame({
    "Feature":     FEATURE_COLS,
    "Coefficient": lr_model.coef_[0],
    "AbsValue":    np.abs(lr_model.coef_[0]),
}).sort_values("AbsValue", ascending=False)

print("Logistic Regression Coefficients (sorted by |value|):")
print(coef_df[["Feature", "Coefficient"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors  = ["#4ade80" if c > 0 else "#f87171" for c in coef_df["Coefficient"]]
bars    = ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor="none")
ax.axvline(0, color="white", linewidth=0.8, alpha=0.4)
ax.set_xlabel("Coefficient", color="#94a3b8")
ax.set_title("LR Feature Coefficients\n(green = increases success probability)", color="#e2e8f0")
ax.set_facecolor("#0f172a")
fig.patch.set_facecolor("#0a0f1e")
ax.tick_params(colors="#94a3b8")
ax.spines[:].set_color("#1e293b")
green_p = mpatches.Patch(color="#4ade80", label="Positive (↑ success)")
red_p   = mpatches.Patch(color="#f87171", label="Negative (↓ success)")
ax.legend(handles=[green_p, red_p], facecolor="#1e293b", labelcolor="#e2e8f0")
plt.tight_layout()
plt.savefig(MODEL_DIR / "feature_coefficients.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── GBM Feature importances ───────────────────────────────────────────────────
gbm_model   = gbm_pipeline.named_steps["gbm"]
imp_df      = pd.DataFrame({
    "Feature":    FEATURE_COLS,
    "Importance": gbm_model.feature_importances_,
}).sort_values("Importance", ascending=False)

print("GBM Feature Importances:")
print(imp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df["Feature"], imp_df["Importance"], color="#6366f1", edgecolor="none")
ax.set_xlabel("Importance", color="#94a3b8")
ax.set_title("GBM Feature Importances", color="#e2e8f0")
ax.set_facecolor("#0f172a")
fig.patch.set_facecolor("#0a0f1e")
ax.tick_params(colors="#94a3b8")
ax.spines[:].set_color("#1e293b")
plt.tight_layout()
plt.savefig(MODEL_DIR / "gbm_importances.png", dpi=120, bbox_inches="tight")
plt.show()

## 6 · Save Trained Models

In [ ]:
def save_models(lr_pipe, gbm_pipe, model_dir: Path) -> dict:
    """
    Persist both pipelines and a metadata manifest to disk.
    Returns the manifest dict for logging / inspection.
    """
    lr_path  = model_dir / "lr_model.pkl"
    gbm_path = model_dir / "gbm_model.pkl"
    meta_path = model_dir / "model_metadata.json"

    with open(lr_path,  "wb") as f: pickle.dump(lr_pipe,  f)
    with open(gbm_path, "wb") as f: pickle.dump(gbm_pipe, f)

    metadata = {
        "trained_at":        datetime.utcnow().isoformat() + "Z",
        "n_train_samples":   int(len(X_train)),
        "n_test_samples":    int(len(X_test)),
        "feature_cols":      FEATURE_COLS,
        "lr_params":         LR_PARAMS,
        "gbm_params":        GBM_PARAMS,
        "test_accuracy":     round(float(accuracy_score(y_test, ens_preds)), 4),
        "test_roc_auc":      round(float(roc_auc_score(y_test, ens_probs)), 4),
        "lr_model_path":     str(lr_path),
        "gbm_model_path":    str(gbm_path),
    }
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved → {lr_path}")
    print(f"Saved → {gbm_path}")
    print(f"Saved → {meta_path}")
    return metadata


metadata = save_models(lr_pipeline, gbm_pipeline, MODEL_DIR)
print("\nMetadata:")
print(json.dumps(metadata, indent=2))

## 7 · Prediction Function

This is the function your FastAPI backend calls for each user × problem pair.  
The models are loaded once at startup and reused across requests.

In [ ]:
# ── Load models from disk (mirrors what FastAPI does at startup) ───────────────
def load_models(model_dir: Path):
    with open(model_dir / "lr_model.pkl",  "rb") as f: lr  = pickle.load(f)
    with open(model_dir / "gbm_model.pkl", "rb") as f: gbm = pickle.load(f)
    return lr, gbm

_lr, _gbm = load_models(MODEL_DIR)
print("Models loaded ✅")

In [ ]:
def predict_success(
    user_rating:      float,
    accuracy:         float,
    solved_problems:  int,
    experience:       str,
    skill_distribution: list,   # [{"name": str, "level": float}]
    difficulty:       str,
    acceptance_rate:  float,    # raw percentage, e.g. 55.6
    problem_tags:     list,     # ["Array", "Dynamic Programming", ...]
    lr_model=_lr,
    gbm_model=_gbm,
) -> dict:
    """
    Predict the probability of a user successfully solving a problem.

    Parameters
    ----------
    user_rating       : Elo-style numeric rating
    accuracy          : overall submission accuracy [0, 1]
    solved_problems   : total unique problems solved
    experience        : "Beginner" | "Intermediate" | "Advanced"
    skill_distribution: user skill vector from MongoDB
    difficulty        : "Easy" | "Medium" | "Hard"
    acceptance_rate   : LeetCode acceptance % (0–100)
    problem_tags      : topic tags on the problem
    lr_model          : loaded LR pipeline
    gbm_model         : loaded GBM pipeline

    Returns
    -------
    dict with keys:
        success_probability  float [0, 1]  — ensemble estimate
        lr_probability       float [0, 1]
        gbm_probability      float [0, 1]
        should_warn          bool          — True when ensemble < 0.50
        confidence           str           — "High" | "Medium" | "Low"
        recommendation       str
    """
    skill_match = compute_skill_match(skill_distribution, problem_tags)

    X = np.array([[
        user_rating,
        accuracy,
        solved_problems,
        EXPERIENCE_MAP.get(experience, 1),
        skill_match,
        DIFFICULTY_MAP.get(difficulty, 2),
        acceptance_rate / 100.0,
    ]])

    lr_prob  = float(lr_model.predict_proba(X)[0][1])
    gbm_prob = float(gbm_model.predict_proba(X)[0][1])
    ensemble = (lr_prob + gbm_prob) / 2.0

    agreement  = 1 - abs(lr_prob - gbm_prob)
    confidence = "High" if agreement > 0.85 else ("Medium" if agreement > 0.65 else "Low")

    if ensemble >= 0.75:
        rec = "Great match! This problem aligns well with your current skill level."
    elif ensemble >= 0.50:
        rec = "Achievable with focused effort. Review relevant topics before starting."
    elif ensemble >= 0.30:
        rec = "Challenging — consider prerequisite problems to build foundational skills first."
    else:
        rec = "Significantly above current level. A step-by-step approach is strongly recommended."

    return {
        "success_probability": round(ensemble, 4),
        "lr_probability":      round(lr_prob, 4),
        "gbm_probability":     round(gbm_prob, 4),
        "should_warn":         ensemble < 0.50,
        "confidence":          confidence,
        "recommendation":      rec,
    }

In [ ]:
# ── Smoke tests against known user profiles ───────────────────────────────────
test_cases = [
    {
        "label":        "Expert user — Easy problem",
        "user_rating":  1520, "accuracy": 0.88, "solved_problems": 350,
        "experience":   "Advanced",
        "skill_distribution": [{"name": "Array", "level": 0.9}, {"name": "Hash Table", "level": 0.85}],
        "difficulty":   "Easy",  "acceptance_rate": 55.6, "problem_tags": ["Array", "Hash Table"],
    },
    {
        "label":        "Beginner — Hard problem",
        "user_rating":  1208, "accuracy": 0.43, "solved_problems": 5,
        "experience":   "Beginner",
        "skill_distribution": [{"name": "Array", "level": 0.34}, {"name": "Dynamic Programming", "level": 0.41}],
        "difficulty":   "Hard", "acceptance_rate": 28.3, "problem_tags": ["String", "Dynamic Programming"],
    },
    {
        "label":        "Intermediate — Medium problem (matching skills)",
        "user_rating":  1280, "accuracy": 0.62, "solved_problems": 60,
        "experience":   "Intermediate",
        "skill_distribution": [{"name": "Array", "level": 0.65}, {"name": "Dynamic Programming", "level": 0.55}],
        "difficulty":   "Medium", "acceptance_rate": 50.1, "problem_tags": ["Array", "Dynamic Programming"],
    },
]

print(f"{'Scenario':<45} {'LR':>6} {'GBM':>6} {'Ensemble':>9} {'Warn':>5}")
print("-" * 75)
for tc in test_cases:
    result = predict_success(**{k: v for k, v in tc.items() if k != "label"})
    print(
        f"{tc['label']:<45} "
        f"{result['lr_probability']:>5.1%} "
        f"{result['gbm_probability']:>6.1%} "
        f"{result['success_probability']:>9.1%} "
        f"{'⚠️' if result['should_warn'] else '✅':>5}"
    )

## 8 · Scheduled Retraining Pipeline

The function below is designed to be called on a schedule (e.g. every 24 hours) to retrain the model on the latest data exported from MongoDB.

**Options for scheduling:**
- **APScheduler** (shown here) — runs in-process, ideal for embedding in FastAPI
- **Cron** — call `python retrain.py` on a system cron schedule
- **Celery Beat** — for distributed task queues

In [ ]:
def retrain_pipeline(
    data_dir:  Path = DATA_DIR,
    model_dir: Path = MODEL_DIR,
    min_samples: int = 20,
) -> dict | None:
    """
    Full retraining pipeline. Called by the scheduler or directly.

    Steps
    -----
    1. Load latest JSON exports from data_dir
    2. Build feature dataframe
    3. Skip retraining if fewer than min_samples rows (not enough data)
    4. Fit LR + GBM pipelines
    5. Evaluate on held-out split
    6. Overwrite model .pkl files only if new model is better (or first run)
    7. Write metadata manifest
    8. Return metadata dict for logging

    Returns None if retraining was skipped.
    """
    ts = datetime.utcnow().isoformat()
    print(f"[{ts}Z] Starting retraining pipeline…")

    # 1. Load
    try:
        users_data    = load_json(data_dir / "users.json")
        ixns_data     = load_json(data_dir / "interactions.json")
        problems_data = load_json(data_dir / "display-problems.json")
    except FileNotFoundError as e:
        print(f"[ERROR] {e}")
        return None

    # 2. Features
    df_new = build_feature_dataframe(users_data, ixns_data, problems_data)

    # 3. Guard
    if len(df_new) < min_samples:
        print(f"[SKIP] Only {len(df_new)} samples — need at least {min_samples}. Retraining skipped.")
        return None

    X_new = df_new[FEATURE_COLS].values
    y_new = df_new["label"].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_new, y_new, test_size=0.2, random_state=42, stratify=y_new
    )

    # 4. Fit
    new_lr  = Pipeline([("scaler", StandardScaler()), ("lr",  LogisticRegression(**LR_PARAMS))])
    new_gbm = Pipeline([("scaler", StandardScaler()), ("gbm", GradientBoostingClassifier(**GBM_PARAMS))])
    new_lr.fit(X_tr, y_tr)
    new_gbm.fit(X_tr, y_tr)

    # 5. Evaluate
    lr_p  = new_lr.predict_proba(X_te)[:, 1]
    gbm_p = new_gbm.predict_proba(X_te)[:, 1]
    ens_p = (lr_p + gbm_p) / 2
    new_acc = float(accuracy_score(y_te, (ens_p >= 0.5).astype(int)))
    new_auc = float(roc_auc_score(y_te, ens_p)) if len(np.unique(y_te)) > 1 else 0.0

    # 6. Compare with current model (if exists)
    meta_path = model_dir / "model_metadata.json"
    prev_auc  = 0.0
    if meta_path.exists():
        with open(meta_path) as f:
            prev_meta = json.load(f)
        prev_auc = prev_meta.get("test_roc_auc", 0.0)

    if new_auc >= prev_auc - 0.02:   # allow 2% tolerance — always update on tie/improvement
        with open(model_dir / "lr_model.pkl",  "wb") as f: pickle.dump(new_lr,  f)
        with open(model_dir / "gbm_model.pkl", "wb") as f: pickle.dump(new_gbm, f)
        action = "UPDATED"
    else:
        action = "KEPT_PREVIOUS (new model performed worse)"

    # 7. Metadata
    meta = {
        "trained_at":       datetime.utcnow().isoformat() + "Z",
        "action":           action,
        "n_train_samples":  int(len(X_tr)),
        "n_test_samples":   int(len(X_te)),
        "test_accuracy":    round(new_acc, 4),
        "test_roc_auc":     round(new_auc, 4),
        "prev_roc_auc":     round(prev_auc, 4),
        "feature_cols":     FEATURE_COLS,
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"[{action}] Accuracy={new_acc:.3f}  ROC-AUC={new_auc:.3f}  (prev AUC={prev_auc:.3f})")
    return meta

In [ ]:
# ── Test the retraining function manually ─────────────────────────────────────
result = retrain_pipeline(DATA_DIR, MODEL_DIR)
if result:
    print("\nRetraining result:")
    print(json.dumps(result, indent=2))

In [ ]:
# ── APScheduler: schedule retraining every 24 hours ───────────────────────────
# This block is intended to run inside your FastAPI startup event.
# The scheduler runs in a background thread — it will NOT block the event loop.

if SCHEDULER_AVAILABLE:
    scheduler = BackgroundScheduler()

    scheduler.add_job(
        func=retrain_pipeline,
        trigger="interval",
        hours=24,
        kwargs={"data_dir": DATA_DIR, "model_dir": MODEL_DIR},
        id="retrain_success_predictor",
        name="Retrain Success Probability Model",
        replace_existing=True,
        # next_run_time=datetime.utcnow()   # uncomment to run immediately on start
    )

    scheduler.start()
    print("Scheduler started ✅")
    print(f"Next run: {scheduler.get_job('retrain_success_predictor').next_run_time}")

    # ── To embed in FastAPI ────────────────────────────────────────────────────
    # from contextlib import asynccontextmanager
    # from fastapi import FastAPI
    #
    # @asynccontextmanager
    # async def lifespan(app: FastAPI):
    #     scheduler.start()
    #     yield
    #     scheduler.shutdown()
    #
    # app = FastAPI(lifespan=lifespan)

else:
    print("APScheduler not available — install with: pip install apscheduler")
    print("Alternatively, run this as a cron job:")
    print("  0 3 * * * /usr/bin/python3 /path/to/retrain.py")

In [ ]:
# ── Reload models after retraining (for hot-swapping in FastAPI) ───────────────
# Call this after retrain_pipeline() completes to update the in-memory models
# without restarting the server.

def reload_models(model_dir: Path = MODEL_DIR):
    """
    Hot-reload the .pkl files into module-level variables.
    Thread-safe for read-heavy workloads (GIL protects the assignment).
    For write-heavy production use, consider a threading.Lock.
    """
    global _lr, _gbm
    _lr, _gbm = load_models(model_dir)
    print(f"[{datetime.utcnow().isoformat()}Z] Models hot-reloaded ✅")

reload_models()

## 9 · Summary & Next Steps

### What's implemented
- ✅ Feature engineering joining users × interactions × problems  
- ✅ Logistic Regression + Gradient Boosting ensemble  
- ✅ Stratified cross-validation + held-out test evaluation  
- ✅ Feature importance & coefficient plots  
- ✅ `predict_success()` function ready for FastAPI integration  
- ✅ `retrain_pipeline()` with model comparison (keeps best model)  
- ✅ APScheduler-based 24-hour retraining loop  
- ✅ Hot-reload after retraining (no server restart needed)  

### Improvements as data grows
| Interaction Count | Recommended Change |
|---|---|
| 50–200 | Current setup — LR + GBM ensemble |
| 200–1 000 | Add `timeTakenSeconds` as a feature; tune GBM depth |
| 1 000–5 000 | Switch to XGBoost; add per-topic skill decay over time |
| 5 000+ | Consider a lightweight neural net or LightGBM with full DKT integration |

### MongoDB export for retraining
```js
// Run in mongosh to export fresh JSON data files:
mongoexport --db codeoptimizer --collection users        --out data/users.json        --jsonArray
mongoexport --db codeoptimizer --collection interactions --out data/interactions.json --jsonArray
mongoexport --db codeoptimizer --collection problems     --out data/display-problems.json --jsonArray
```